In [1]:
# Cell 1 - imports and constants
# carrying forward all decisions made during exploration

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# paths
DATA_RAW       = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(exist_ok=True)

# column names
index_names   = ['unit_number', 'time_cycles']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names  = ['s_{}'.format(i+1) for i in range(21)]
col_names     = index_names + setting_names + sensor_names

# dropping these — either constant or near-constant
SENSORS_TO_DROP = ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']

# 14 sensors kept after exploration
FEATURE_COLS = ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9',
                's_11', 's_12', 's_13', 's_14', 's_15',
                's_17', 's_20', 's_21']

# RUL cap — degradation only visible in last 125 cycles
RUL_CAP = 125

# sequence length — how many cycles to look back
# using 30 cycles as a good balance between context and data size
SEQUENCE_LENGTH = 30

print("constants loaded:")
print(f"  features:        {len(FEATURE_COLS)}")
print(f"  RUL cap:         {RUL_CAP}")
print(f"  sequence length: {SEQUENCE_LENGTH}")

constants loaded:
  features:        14
  RUL cap:         125
  sequence length: 30


In [2]:
# Cell 2 - loading and cleaning the data

# load all three files
df_train = pd.read_csv(
    DATA_RAW / 'train_FD001.txt',
    sep='\s+', header=None, names=col_names
)
df_test = pd.read_csv(
    DATA_RAW / 'test_FD001.txt',
    sep='\s+', header=None, names=col_names
)
df_rul = pd.read_csv(
    DATA_RAW / 'RUL_FD001.txt',
    sep='\s+', header=None, names=['RUL']
)

# drop constant and near-constant sensors
df_train = df_train.drop(columns=SENSORS_TO_DROP + setting_names)
df_test  = df_test.drop(columns=SENSORS_TO_DROP + setting_names)

# add RUL to training data
df_train['RUL'] = (
    df_train.groupby('unit_number')['time_cycles']
    .transform('max') - df_train['time_cycles']
)

# cap RUL at 125
df_train['RUL'] = df_train['RUL'].clip(upper=RUL_CAP)

# add RUL to test data
# test data ends before failure — true RUL is in df_rul
# we add the true RUL to the last cycle of each test engine
df_test['RUL'] = 0
for i, unit in enumerate(df_test['unit_number'].unique()):
    mask = df_test['unit_number'] == unit
    df_test.loc[mask, 'RUL'] = df_rul.iloc[i]['RUL']

# cap test RUL too
df_test['RUL'] = df_test['RUL'].clip(upper=RUL_CAP)

print("data loaded and cleaned:")
print(f"  training: {df_train.shape[0]:,} rows, {df_train.shape[1]} columns")
print(f"  test:     {df_test.shape[0]:,} rows, {df_test.shape[1]} columns")
print(f"\ntraining columns: {list(df_train.columns)}")
print(f"\nRUL range in training: {df_train['RUL'].min()} to {df_train['RUL'].max()}")
print(f"RUL range in test:     {df_test['RUL'].min()} to {df_test['RUL'].max()}")

data loaded and cleaned:
  training: 20,631 rows, 17 columns
  test:     13,096 rows, 17 columns

training columns: ['unit_number', 'time_cycles', 's_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21', 'RUL']

RUL range in training: 0 to 125
RUL range in test:     7 to 125


In [3]:
# Cell 3 - scaling the sensor data
# using RobustScaler — handles outliers better than MinMaxScaler
# fitting ONLY on training data — never on test data to avoid leakage

scaler = RobustScaler()

# fit and transform training sensors
df_train[FEATURE_COLS] = scaler.fit_transform(df_train[FEATURE_COLS])

# transform test sensors using the same scaler fitted on training
df_test[FEATURE_COLS] = scaler.transform(df_test[FEATURE_COLS])

# save scaler — needed later in the API to scale incoming requests
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("scaling done:")
print(f"  scaler fitted on training data only")
print(f"  scaler saved to models/scaler.pkl")
print(f"\ntraining data after scaling (first 3 rows):")
print(df_train[FEATURE_COLS].head(3).round(3))
print(f"\ntraining sensor stats after scaling:")
print(df_train[FEATURE_COLS].describe().round(3).loc[['mean', 'std', 'min', 'max']])

scaling done:
  scaler fitted on training data only
  scaler saved to models/scaler.pkl

training data after scaling (first 3 rows):
     s_2    s_3    s_4    s_7    s_8    s_9   s_11   s_12  s_13   s_14   s_15  \
0 -1.215 -0.049 -0.610  0.767 -0.333 -0.887 -0.114  0.182  -0.7 -0.127 -0.383   
1 -0.726  0.212 -0.402  0.258 -0.556 -1.017 -0.057  0.808  -0.2 -0.601 -0.140   
2 -0.430 -0.260 -0.315  0.683 -0.111 -0.473 -0.686  0.949  -0.6 -0.485 -0.416   

   s_17  s_20   s_21  
0  -0.5  0.92  0.835  
1  -0.5  0.68  0.867  
2  -1.5  0.48  0.319  

training sensor stats after scaling:
        s_2    s_3    s_4    s_7    s_8     s_9   s_11   s_12   s_13    s_14  \
mean  0.061  0.052  0.073 -0.060  0.074   0.281  0.089 -0.067  0.062   0.213   
std   0.741  0.755  0.738  0.738  0.789   1.353  0.763  0.745  0.719   1.266   
min  -2.119 -2.347 -2.115 -2.992 -2.111  -2.385 -1.886 -2.818 -2.100  -2.695   
max   2.800  3.302  2.743  2.183  5.222  11.270  2.914  1.919  4.700  10.168   

       s_15

In [4]:
# Cell 4 - creating sequences for the GRU-LSTM model
# the model needs to see a window of past cycles to predict RUL
# for each cycle, we take the previous SEQUENCE_LENGTH cycles as input

def create_sequences(df, sequence_length, feature_cols):
    """
    for each engine, slide a window of sequence_length cycles
    and create one sample per valid window position
    """
    X, y = [], []
    
    for unit in df['unit_number'].unique():
        engine_data = df[df['unit_number'] == unit].reset_index(drop=True)
        
        # need at least sequence_length cycles to make one sample
        if len(engine_data) < sequence_length:
            continue
            
        for i in range(len(engine_data) - sequence_length + 1):
            # input: sequence_length cycles of sensor readings
            X.append(engine_data[feature_cols].iloc[i:i+sequence_length].values)
            # target: RUL at the last cycle of the window
            y.append(engine_data['RUL'].iloc[i+sequence_length-1])
    
    return np.array(X), np.array(y)

# create training sequences
X_train, y_train = create_sequences(df_train, SEQUENCE_LENGTH, FEATURE_COLS)

# for test data — we only want the LAST sequence per engine
# because we're predicting RUL at the last observed cycle
X_test, y_test = [], []
for unit in df_test['unit_number'].unique():
    engine_data = df_test[df_test['unit_number'] == unit]
    if len(engine_data) >= SEQUENCE_LENGTH:
        # take last SEQUENCE_LENGTH cycles
        X_test.append(engine_data[FEATURE_COLS].iloc[-SEQUENCE_LENGTH:].values)
    else:
        # pad with zeros if engine has fewer cycles than sequence length
        pad_length = SEQUENCE_LENGTH - len(engine_data)
        padded = np.pad(
            engine_data[FEATURE_COLS].values,
            ((pad_length, 0), (0, 0)),
            mode='constant'
        )
        X_test.append(padded)
    y_test.append(df_rul.iloc[unit-1]['RUL'])

X_test  = np.array(X_test)
y_test  = np.array(y_test)

print("sequences created:")
print(f"  X_train shape: {X_train.shape}  → (samples, timesteps, features)")
print(f"  y_train shape: {y_train.shape}  → (samples,)")
print(f"  X_test shape:  {X_test.shape}   → (engines, timesteps, features)")
print(f"  y_test shape:  {y_test.shape}   → (engines,)")
print(f"\ny_train range: {y_train.min():.1f} to {y_train.max():.1f}")
print(f"y_test range:  {y_test.min():.1f} to {y_test.max():.1f}")

sequences created:
  X_train shape: (17731, 30, 14)  → (samples, timesteps, features)
  y_train shape: (17731,)  → (samples,)
  X_test shape:  (100, 30, 14)   → (engines, timesteps, features)
  y_test shape:  (100,)   → (engines,)

y_train range: 0.0 to 125.0
y_test range:  7.0 to 145.0


In [5]:
# Cell 5 - saving sequences and model config
# saving everything needed for training and the API later

# save sequences
np.save('../data/processed/X_train.npy', X_train)
np.save('../data/processed/y_train.npy', y_train)
np.save('../data/processed/X_test.npy',  X_test)
np.save('../data/processed/y_test.npy',  y_test)

# save model config — everything the API needs to know
model_config = {
    'feature_cols':      FEATURE_COLS,
    'sequence_length':   SEQUENCE_LENGTH,
    'n_features':        len(FEATURE_COLS),
    'rul_cap':           RUL_CAP,
    'sensors_dropped':   SENSORS_TO_DROP,
}

with open('../models/model_config.pkl', 'wb') as f:
    pickle.dump(model_config, f)

print("saved:")
print(f"  data/processed/X_train.npy  — {X_train.nbytes / 1e6:.1f} MB")
print(f"  data/processed/y_train.npy  — {y_train.nbytes / 1e6:.1f} MB")
print(f"  data/processed/X_test.npy   — {X_test.nbytes / 1e6:.1f} MB")
print(f"  data/processed/y_test.npy   — {y_test.nbytes / 1e6:.1f} MB")
print(f"  models/model_config.pkl")
print(f"  models/scaler.pkl")
print(f"\nmodel config:")
for k, v in model_config.items():
    print(f"  {k}: {v}")

saved:
  data/processed/X_train.npy  — 59.6 MB
  data/processed/y_train.npy  — 0.1 MB
  data/processed/X_test.npy   — 0.3 MB
  data/processed/y_test.npy   — 0.0 MB
  models/model_config.pkl
  models/scaler.pkl

model config:
  feature_cols: ['s_2', 's_3', 's_4', 's_7', 's_8', 's_9', 's_11', 's_12', 's_13', 's_14', 's_15', 's_17', 's_20', 's_21']
  sequence_length: 30
  n_features: 14
  rul_cap: 125
  sensors_dropped: ['s_1', 's_5', 's_6', 's_10', 's_16', 's_18', 's_19']
